# NB08 — Rust: Ownership, Borrowing, and Lifetimes

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

The borrow checker is the part of Rust that confuses new learners most. It is also the feature that makes Rust uniquely safe: no dangling pointers, no data races, no use-after-free — guaranteed at compile time, not by a runtime GC.

To write Rust, to read noodles source code, to understand why a PyO3 function signature looks the way it does — you must understand ownership and borrowing. This notebook is the deep dive.

---

## 2. Intuition

**Ownership:** Every value in Rust has exactly one owner. When the owner goes out of scope, the value is dropped (freed). No GC, no reference counting.

**Borrowing:** You can lend a value temporarily — either shared (`&T`, read-only, many borrows allowed) or exclusive (`&mut T`, read-write, only one borrow allowed). While a borrow is active, the owner cannot drop or mutate the value. The borrow checker enforces all of this at compile time.

**Lifetimes:** A lifetime `'a` is the borrow checker's name for a scope. When a function returns a reference, the compiler needs to know: "how long is this reference valid?" Lifetime annotations tell the compiler the relationship between input and output reference durations.

---

## 3. Biological background

A FASTA parser in Rust might look like:

```rust
fn first_record<'a>(content: &'a str) -> &'a str {
    // Returns a slice of `content` — valid as long as `content` is
    content.lines().nth(1).unwrap_or("")
}
```

The `'a` lifetime says: "the returned `&str` is a borrow from `content`, so it can't outlive `content`." The borrow checker ensures this. If `content` is dropped before the returned slice is used, it's a compile error — not a runtime segfault.

This is why needletail can parse 50 GB FASTQ files with zero-copy string slices — every returned `&str` is a slice of the buffer, valid exactly as long as the buffer exists.

---

## 4. Mathematical explanation

### The three ownership rules

1. Each value in Rust has exactly one **owner** variable.
2. There can only be one owner at a time.
3. When the owner goes out of scope, the value is **dropped** (freed).

### The borrow rules

At any point in time, for a given value, you can have either:
- Any number of **immutable references** (`&T`) — read-only, no writes allowed through them
- Exactly **one mutable reference** (`&mut T`) — exclusive read-write access

These are mutually exclusive: you cannot have a `&T` and a `&mut T` to the same value at the same time.

This is equivalent to a **readers-writers lock** enforced at compile time, with no runtime overhead.

---

## 5. Computational explanation

```rust
// Ownership: move
let s1 = String::from("ATCG");
let s2 = s1;           // s1 moved into s2
// println!("{}", s1); // ERROR: value borrowed after move

// Borrowing: shared reference (&T)
let s3 = String::from("ATCG");
let r1 = &s3;          // immutable borrow
let r2 = &s3;          // OK: multiple immutable borrows
println!("{} {}", r1, r2);  // both valid

// Borrowing: mutable reference (&mut T)
let mut s4 = String::from("AT");
let r3 = &mut s4;      // mutable borrow
r3.push_str("CG");    // OK
// let r4 = &s4;       // ERROR: s4 also mutably borrowed

// Lifetimes
fn longest<'a>(x: &'a str, y: &'a str) -> &'a str {
    if x.len() > y.len() { x } else { y }
    // 'a: the return lives as long as the shorter of x and y
}
```

---

## 6. Python implementation — analogies and borrow checker error explanations

In [ ]:
import sys
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("NB08: Rust Ownership, Borrowing, and Lifetimes")
print("Python reference counting provides an analogy for understanding ownership.")

In [ ]:
# --- Python reference counting as an analogy for Rust ownership ---

# In Python, variables are names that refer to objects.
# Multiple names can refer to the same object (shared ownership via refcount).

s1 = "ATCG"  # refcount(ATCG) = 1
s2 = s1       # refcount(ATCG) = 2 — both s1 and s2 refer to the same str
del s1        # refcount(ATCG) = 1 — s2 still valid
del s2        # refcount(ATCG) = 0 — string freed

print("Python: shared reference counting")
a = [1, 2, 3]
b = a          # b and a refer to the SAME list object
b.append(4)
print(f"  a = {a}  (modified through b — same object!)")
print(f"  id(a) == id(b): {id(a) == id(b)}")

# In Rust, assignment MOVES for non-Copy types (String, Vec).
# The Python equivalent would be: b = a; a = None  (a is no longer usable)
# Rust enforces this at compile time — Python doesn't enforce it at all.
print()
print("Rust equivalent (conceptual Python):")
a = [1, 2, 3]
b = a          # In Rust: a MOVES into b
a = None       # Rust would give you a compile error if you tried to use a after this move
print(f"  After move: a = {a} (invalid — do not use)")
print(f"              b = {b} (new owner)")

In [ ]:
# --- Common borrow checker errors and their fixes ---

error_1 = '''
// ERROR 1: Use after move
let s1 = String::from("ATCG");
let s2 = s1;           // s1 moved into s2
println!("{}", s1);   // ERROR: value borrowed here after move
//                     borrow of moved value: `s1`

// FIX: clone if you need both
let s1 = String::from("ATCG");
let s2 = s1.clone();   // explicit heap copy — s1 still valid
println!("{} {}", s1, s2);   // OK

// FIX 2: use a reference instead of moving
let s1 = String::from("ATCG");
let s2 = &s1;          // borrow, not move — s1 still owns the data
println!("{} {}", s1, s2);   // OK
'''

error_2 = '''
// ERROR 2: Mutate while borrowed
let mut v = vec![1, 2, 3];
let first = &v[0];     // immutable borrow of v
v.push(4);             // ERROR: cannot borrow `v` as mutable because also
                       //         borrowed as immutable
println!("{}", first); // first might dangle after push (reallocation!)

// FIX: end the immutable borrow before mutating
let mut v = vec![1, 2, 3];
let first_value = v[0];  // Copy the value out (i32 is Copy)
v.push(4);               // OK: no active borrow
println!("{}", first_value);  // OK
'''

error_3 = '''
// ERROR 3: Dangling reference (lifetime violation)
fn get_sequence() -> &str {   // ERROR: missing lifetime specifier
    let seq = String::from("ATCG");  // seq is local to this function
    &seq                              // would return reference to freed memory!
}                                    // seq dropped here

// FIX: return an owned String, not a reference
fn get_sequence_fixed() -> String {
    String::from("ATCG")    // ownership transferred to caller
}

// FIX 2: take a String parameter and return a slice of it
fn first_kmer<\'a>(seq: &\'a str, k: usize) -> &\'a str {
    &seq[..k]    // slice of the input — valid as long as seq is valid
}
'''

print("Common borrow checker errors:")
print("\nERROR 1: Use after move")
print(error_1)
print("ERROR 2: Mutate while borrowed")
print(error_2)
print("ERROR 3: Dangling reference")
print(error_3)

In [ ]:
# --- Python analogy for each Rust concept ---

print("=" * 60)
print("Python analogies for Rust ownership concepts")
print("=" * 60)

# 1. Immutable reference (&T): like passing a read-only view
def print_gc(seq: str) -> None:  # str in Python is immutable — natural &str
    gc = sum(1 for b in seq if b in 'GC')
    print(f"  GC content: {gc/len(seq):.2f}")

my_seq = "ATCGCGAT"
print_gc(my_seq)  # seq is 'borrowed' for the duration of the call
print(f"  my_seq still valid: '{my_seq}'")

# 2. Mutable reference (&mut T): like passing an object you can modify
def append_reverse(seq_list: list, item: str) -> None:  # list is mutable
    seq_list.append(item[::-1])  # modifies the original list

seqs = ["ATCG"]
append_reverse(seqs, "GCTA")
print(f"  After mutable borrow: {seqs}")

# 3. sys.getrefcount as a proxy for ownership
s = "UNIQUE_STRING_12345"
refs = sys.getrefcount(s)  # includes the argument to getrefcount itself
print(f"\n  Reference count of s: {refs} (includes getrefcount's own reference)")
t = s
refs2 = sys.getrefcount(s)
print(f"  After t = s: reference count = {refs2} (increased by 1)")
print("  In Rust, s would be MOVED into t — s would be invalid.")

In [ ]:
# --- Lifetimes: intuition through Python analogy ---

rust_lifetime_code = '''
// Lifetimes: explicit scope annotations

// Without lifetime annotation (Rust infers this automatically for simple cases)
fn first_base(seq: &str) -> char {
    seq.chars().next().unwrap()
}  // Return is char (owned), not a reference — no lifetime needed

// With explicit lifetime: return borrows from one of the inputs
fn longer_seq<\'a>(a: &\'a str, b: &\'a str) -> &\'a str {
    // The returned reference is valid as long as BOTH a and b are valid
    // \'a = the SHORTER of the two input lifetimes
    if a.len() >= b.len() { a } else { b }
}

// Lifetime in struct: the struct cannot outlive the data it references
struct FastaRecord<\'a> {
    id: &\'a str,       // slice of the input buffer
    sequence: &\'a str, // slice of the input buffer
}  // FastaRecord is only valid while the buffer (\'a) is alive

// This is how zero-copy parsers work:
// The parsed records ARE slices of the original buffer.
// No string allocation per record — just pointer + length.
'''

print("Rust lifetimes:")
print(rust_lifetime_code)

# Python analogy: views into a buffer (numpy as the closest equivalent)
import numpy as np
buffer = np.array([65, 84, 67, 71, 65, 84, 67, 71], dtype=np.uint8)  # ATCGATCG
view1 = buffer[:4]   # slice = a view, not a copy
view2 = buffer[4:]   # another view
print("Python numpy view analogy:")
print(f"  buffer: {buffer}")
print(f"  view1 (slice of buffer): {view1}  (shares memory: {np.shares_memory(buffer, view1)})")
print(f"  view2: {view2}")
print("\nIn Rust, FastaRecord slices would be like numpy views:")
print("  - They reference into the original buffer")
print("  - They CANNOT outlive the buffer (Rust enforces this; numpy doesn't)")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Ownership transfer diagram
ax = axes[0]
ax.axis('off')
own_transfer = (
    "Ownership Transfer\n\n"
    "let s1 = String::from(\"ATCG\");\n"
    "\n"
    "  s1  ──owns──► [heap: ATCG]\n"
    "\n"
    "let s2 = s1;  // MOVE\n"
    "\n"
    "  s1  ──✗ (invalid, moved)\n"
    "  s2  ──owns──► [heap: ATCG]\n"
    "\n"
    "} // end of scope: s2 dropped\n"
    "  [heap: ATCG]  freed\n\n"
    "vs. Clone:\n"
    "let s2 = s1.clone();\n"
    "  s1  ──owns──► [heap: ATCG]  (original)\n"
    "  s2  ──owns──► [heap: ATCG]  (new copy)"
)
ax.text(0.02, 0.98, own_transfer, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eaf2f8', alpha=0.8))
ax.set_title('Ownership: Move vs Clone', fontweight='bold')

# Panel 2: Borrow vs Move
ax = axes[1]
ax.axis('off')
borrow_text = (
    "Borrowing Rules\n\n"
    "let s = String::from(\"ATCG\");\n\n"
    "// Shared borrows (&T): many OK\n"
    "let r1 = &s;   // read-only\n"
    "let r2 = &s;   // read-only — OK\n"
    "println!({r1} {r2});  // both valid\n\n"
    "// Mutable borrow (&mut T): exactly one\n"
    "let mut s2 = String::from(\"AT\");\n"
    "let rm = &mut s2;\n"
    "rm.push_str(\"CG\");  // OK\n"
    "// let rm2 = &mut s2;  ERROR\n"
    "// let r3  = &s2;      ERROR\n\n"
    "Rule: at any time,\n"
    "  EITHER: any number of &T\n"
    "  OR:     exactly one &mut T\n"
    "  NEVER both simultaneously"
)
ax.text(0.02, 0.98, borrow_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#fef9e7', alpha=0.8))
ax.set_title('Borrowing Rules\n(&T vs &mut T)', fontweight='bold')

# Panel 3: Lifetime scope diagram
ax = axes[2]
ax.axis('off')
lifetime_text = (
    "Lifetime Scope Diagram\n\n"
    "fn longest<\'a>(x: &\'a str, y: &\'a str) -> &\'a str\n\n"
    "Scope:  {\n"
    "  let s1 = String::from(\"long\");\n"
    "  let result;\n"
    "  {\n"
    "    let s2 = String::from(\"xy\");\n"
    "    result = longest(&s1, &s2);\n"
    "    println!(\"{result}\");  // OK: s1, s2 valid\n"
    "  }  // s2 dropped here\n"
    "  // println!(\"{result}\");  // ERROR\n"
    "  // result might be &s2 which is freed\n"
    "}\n\n"
    "The compiler tracks that \'a is\n"
    "the SHORTER of the two input\n"
    "lifetimes (x and y).\n"
    "Using result outside that scope\n"
    "= compile error."
)
ax.text(0.02, 0.98, lifetime_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eafaf1', alpha=0.8))
ax.set_title('Lifetime Scope:\nfn longest<\'a>(...)', fontweight='bold')

plt.tight_layout()
plt.savefig('../datasets/nb08_ownership.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Python RC model:** Write a Python function that demonstrates the reference-counting analog of each Rust ownership operation: move, clone, immutable borrow, mutable borrow. Use `sys.getrefcount` to show the count at each step.

2. **Borrow error diagnosis:** For each Rust snippet below (shown as a string), identify which borrow rule is violated and write the corrected version:
   - `let v = vec![1,2,3]; let r = &v[0]; v.push(4); println!("{}", r);`
   - `let s = String::from("x"); let r = &mut s; let r2 = &mut s;`

3. **Python dangling reference analog:** Python doesn't allow dangling references because it's garbage collected. Write a Python example that *would* be a dangling reference in C (a function returns a pointer to a local variable), and explain why Python's GC prevents it but C does not.

4. **FastaRecord with lifetime (Python):** Implement a Python class `FastaView` that takes a bytes object as a buffer and stores slices (`memoryview`) of it rather than copying. Verify that the slices share memory with the original buffer using `memoryview.tobytes()`.

## 12. Reflection

*Write here: In your own words, what is a lifetime annotation telling the Rust compiler? Why does Rust need them when Python doesn't? What would go wrong if Rust didn't have them?*

---

## References

- The Rust Book, Chapter 4: Understanding Ownership — https://doc.rust-lang.org/book/ch04-00-understanding-ownership.html
- The Rust Book, Chapter 10.3: Lifetimes — https://doc.rust-lang.org/book/ch10-03-lifetime-syntax.html
- Rust Reference: Ownership and Lifetimes — https://doc.rust-lang.org/reference/ownership.html

## Future work / open questions

- Rust's `Rc<T>` and `Arc<T>` provide Python-like reference counting inside Rust when you genuinely need shared ownership. What is the difference between `Rc<T>` (single-threaded) and `Arc<T>` (thread-safe)?
- NLL (Non-Lexical Lifetimes) in Rust 2018+ allows borrows to end earlier than the end of the enclosing scope. How does this reduce the number of borrow checker errors in practice?